# ARCHIVED — standalone background preprocessing

Superseded: preprocessing is now inline in `notebooks/01_generate_grid.ipynb` as
`load_chest()`, so there is no separate step to forget or to let drift out of sync with the
real-image path.

The three decisions it introduced are still in force, and still matter:

1. **Crop to square, never squash.** Aspect ratios in NODE21 run 0.820 to 1.219, so
   resizing straight to square stretched the most portrait chests by over 20%.
2. **Percentile normalisation, not min-max.** Several images have pixels saturated at both
   0 and 65519; min-max anchors on those and compresses the actual tissue range.
3. **Record what was done.** Original size, crop offsets, scale and pixel format go into
   the CSV rather than living as default arguments in a function.

Reprocess the 50 background X-rays: crop to square, percentile-normalise, CLAHE, 512x512.

WHAT CHANGED FROM THE OLD PREPROCESSOR

  1. CROP, not squash. The old one did cv2.resize(img, (512,512)) with no regard for
     shape. Aspect ratios in this set run 0.820 to 1.219, so the most portrait images had
     their width stretched by over 20%. 31 of the 50 are non-square.

  2. PERCENTILE normalisation, not min-max. Several images have pixels clipped at both
     0 and 65519. Min-max anchors on those saturated extremes and squeezes the actual
     tissue range into the middle. Clipping to the 1st-99th percentile spreads tissue
     across the full range instead.

  3. RECORDS WHAT IT DID. Original size, crop offsets, scale, pixel format -- written to
     backgrounds.csv rather than living only as a default argument in a function. Half
     the reason the old coordinate bug survived is that nothing downstream knew what
     size or space it was working in.

OUTPUT
  <OUT_DIR>/cXXXX_processed.npy   float32, 512x512, range 0-1
  <OUT_DIR>/backgrounds.csv       one row per background
  <OUT_DIR>/crop_check.png        before/after grid so you can see what was cut

In [ ]:
import json
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import SimpleITK as sitk
import matplotlib.pyplot as plt

In [ ]:
BASE      = Path('/content/drive/MyDrive/.../node21/cxr_images')   # <-- set
MHA_DIR   = BASE / 'original_data/images'
SELECTION = BASE / 'clean_backgrounds/clean_background_selection.json'
OUT_DIR   = BASE / 'backgrounds_v3'          # NEW folder -- do not overwrite the old

TARGET      = 512
PCT_LO, PCT_HI = 1, 99        # clip here instead of min/max
CLAHE_CLIP  = 2.0
CLAHE_GRID  = (8, 8)

OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ---------------------------------------------------------------- steps

def load_mha(path):
    a = sitk.GetArrayFromImage(sitk.ReadImage(str(path))).astype(np.float32)
    return a.squeeze() if a.ndim == 3 else a


def guess_format(a):
    m = a.max()
    return '8-bit' if m <= 255 else '12-bit' if m <= 4095 else '16-bit'


def crop_square(a):
    """Centre-crop to square. Returns (cropped, x_offset, y_offset, side)."""
    h, w = a.shape
    side = min(h, w)
    x0 = (w - side) // 2
    y0 = (h - side) // 2
    return a[y0:y0 + side, x0:x0 + side], x0, y0, side


def normalise(a, lo_pct=PCT_LO, hi_pct=PCT_HI):
    """Clip to percentiles, then scale to 0-1. Robust to saturated pixels."""
    lo, hi = np.percentile(a, [lo_pct, hi_pct])
    if hi - lo < 1e-6:
        return np.zeros_like(a), float(lo), float(hi)
    return np.clip((a - lo) / (hi - lo), 0, 1).astype(np.float32), float(lo), float(hi)


def clahe(a01):
    """Local contrast enhancement. Expects and returns 0-1 float."""
    c = cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=CLAHE_GRID)
    return (c.apply((a01 * 255).astype(np.uint8)).astype(np.float32) / 255.0)


def process(path):
    raw = load_mha(path)
    oh, ow = raw.shape
    fmt = guess_format(raw)

    cropped, x0, y0, side = crop_square(raw)
    norm, lo, hi = normalise(cropped)
    enhanced = clahe(norm)
    small = cv2.resize(enhanced, (TARGET, TARGET), interpolation=cv2.INTER_AREA)
    small = np.clip(small, 0, 1).astype(np.float32)     # guard interpolation drift

    meta = {
        'orig_width': ow, 'orig_height': oh,
        'aspect': round(ow / oh, 4),
        'pixel_format': fmt,
        'raw_min': float(raw.min()), 'raw_max': float(raw.max()),
        'clip_lo': round(lo, 1), 'clip_hi': round(hi, 1),
        'crop_x0': x0, 'crop_y0': y0, 'crop_side': side,
        'pct_width_cut':  round(100 * (ow - side) / ow, 1),
        'pct_height_cut': round(100 * (oh - side) / oh, 1),
        'scale': round(TARGET / side, 4),
        'proc_width': TARGET, 'proc_height': TARGET,
    }
    return small, meta, raw


# ---------------------------------------------------------------- run

names = [s['img_name'] for s in json.load(open(SELECTION))]
print(f'{len(names)} backgrounds to process\n')

rows, previews = [], []
for i, name in enumerate(names):
    src = MHA_DIR / name
    if not src.exists():
        print(f'[MISSING] {name}'); continue

    img, meta, raw = process(src)

    stem = Path(name).stem                       # c0221
    out  = OUT_DIR / f'{stem}_processed.npy'     # keep the old naming convention
    np.save(out, img)

    rows.append({'background_id': f'bg_{i:02d}', 'node21_img_name': name,
                 'npy_path': str(out), 'pixel_std': round(float(img.std()), 4), **meta})

    if len(previews) < 8:
        previews.append((name, raw, img, meta))

    if meta['pct_height_cut'] > 10 or meta['pct_width_cut'] > 10:
        print(f'  [heavy crop] {name}: cut {meta["pct_width_cut"]}% width, '
              f'{meta["pct_height_cut"]}% height')

bg = pd.DataFrame(rows)
bg.to_csv(OUT_DIR / 'backgrounds.csv', index=False)


# ---------------------------------------------------------------- verify

print(f'\nwrote {len(bg)} files to {OUT_DIR}')
print(f'\npixel formats: {bg.pixel_format.value_counts().to_dict()}')
print(f'crop removed >10% on {((bg.pct_width_cut > 10) | (bg.pct_height_cut > 10)).sum()} images')
print(f'largest cut: {max(bg.pct_width_cut.max(), bg.pct_height_cut.max()):.1f}%')

# the four checks from the checklist
assert len(bg) == len(names),                      'row count mismatch'
assert bg.background_id.nunique() == len(bg),       'duplicate background_id'
shapes = {tuple(np.load(p).shape) for p in bg.npy_path}
assert shapes == {(TARGET, TARGET)}, f'inconsistent shapes: {shapes}'
lo = min(float(np.load(p).min()) for p in bg.npy_path)
hi = max(float(np.load(p).max()) for p in bg.npy_path)
assert 0 <= lo and hi <= 1,                         f'values outside [0,1]: {lo}, {hi}'
assert (bg.pixel_std > 0.05).all(),                 'some images have too little contrast'
print('\nall four checks passed')


# ---------------------------------------------------------------- look at it

fig, axes = plt.subplots(2, len(previews), figsize=(2.6*len(previews), 5.6))
for j, (name, raw, img, m) in enumerate(previews):
    axes[0, j].imshow(raw, cmap='gray')
    axes[0, j].set_title(f'{name}\n{m["orig_width"]}x{m["orig_height"]}  '
                         f'ar {m["aspect"]}', fontsize=7)
    axes[1, j].imshow(img, cmap='gray')
    axes[1, j].set_title(f'cut {m["pct_width_cut"]}%w {m["pct_height_cut"]}%h', fontsize=7)
    axes[0, j].axis('off'); axes[1, j].axis('off')
axes[0, 0].set_ylabel('original'); axes[1, 0].set_ylabel('processed')
plt.suptitle('top: original   bottom: cropped, normalised, 512x512', fontsize=10)
plt.tight_layout(); plt.savefig(OUT_DIR / 'crop_check.png', dpi=120); plt.show()

print('\nNow look at crop_check.png. You are checking that the crop did not cut off')
print('lung apices at the top or costophrenic angles at the bottom. The lung-box edge')
print('test in find_lung_fields.py will catch any it did.')